# Lab: Retrieval -- Finding Similar Items Without Labels

**Bluevision-ai Academy**
*Retrieval: Finding Similar Items Without Labels*

In lecture the answer column vanished, and we found the first question that still survives without labels: what is similar to this? We turned rows into vectors, picked a distance, and built nearest-neighbour retrieval -- then discovered the label's old job (deciding what "similar" means) had quietly landed on us. This lab does all four of those moves yourself, in code, on a small dataset you can read start to finish.

Each part runs the same rhythm the lecture did, just in cells instead of clicks: **Problem** (what we have, what we want, why the obvious tool doesn't reach it) -> **Idea** (the reasoning move, in words, before any code) -> **Build** (the code that embodies exactly that idea) -> **Payoff** (what just happened, and what it means). One idea per cell -- if a cell needs "and" to describe it, it's doing too much.

| Part | What you build | Ties back to |
|---|---|---|
| 1 -- Vectors & Distance | Turn songs into feature vectors; compute Euclidean distance by hand, then with numpy | Slides 5-6 |
| 2 -- Build kNN | A nearest-neighbour retrieval function; query it for "songs like this one" | Slide 7 |
| 3 -- Euclidean vs Cosine | The loud-remaster trick: one metric is fooled by volume, the other isn't | Slide 8 |
| Challenge | Break it on purpose: a wildly different-scaled feature | Slide 8 closing |
| Reflection | Five questions, answered in code comments | -- |

**Dataset:** twelve songs described only by audio features -- tempo, energy, danceability, acousticness, valence -- no genre or mood label anywhere. **Estimated time:** ~2 hours. Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

Colab already ships with everything this lab needs. The cell below just makes sure versions are current, turns on interactive widgets, and loads the deck's own palette so these plots read like the ones you just watched.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade scikit-learn plotly ipywidgets

import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider, fixed

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running on Colab -- local Jupyter with ipywidgets already works

np.random.seed(0)

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#7f8c9a"

## Part 1 -- Data as Vectors, Distance by Hand

**Problem.** Slide 5 made one move: read a row of a table as a point in space. Slide 6 asked the question that move immediately raises -- once two songs are points, what number says how "close" they are? We have a small catalogue of songs described only by audio features, no labels anywhere, and we want a single number that captures "how similar are these two songs" using nothing but arithmetic.

**Ingredients.** Twelve songs, five audio features each (tempo, energy, danceability, acousticness, valence), all on a 0-1 scale, no label column anywhere. One row is a planted trap -- the same song, recorded roughly three times louder -- keep it in mind; it matters a lot in Part 3.

In [2]:
songs = {
    "Sunrise Drive":                   [0.62, 0.71, 0.58, 0.10, 0.66],
    "Neon Nights":                     [0.88, 0.93, 0.81, 0.05, 0.72],
    "Quiet Rain":                      [0.21, 0.18, 0.30, 0.85, 0.25],
    "Midnight Study":                  [0.19, 0.15, 0.22, 0.90, 0.20],
    "Coffee Shop Acoustic":            [0.30, 0.25, 0.35, 0.80, 0.40],
    "Festival Anthem":                 [0.90, 0.95, 0.85, 0.03, 0.80],
    "Highway Chase":                   [0.85, 0.88, 0.60, 0.08, 0.55],
    "Slow Sunday":                     [0.25, 0.20, 0.28, 0.82, 0.30],
    "Club Bassline":                   [0.92, 0.90, 0.88, 0.02, 0.68],
    "Piano Reflection":                [0.15, 0.10, 0.20, 0.92, 0.22],
    "Road Trip Sing-Along":            [0.70, 0.75, 0.65, 0.12, 0.78],
    "Festival Anthem (Loud Remaster)": [2.70, 2.85, 2.55, 0.09, 2.40],
}
feature_names = ["tempo", "energy", "danceability", "acousticness", "valence"]
song_names = list(songs.keys())
X_songs = np.array(list(songs.values()))
X_songs.shape

(12, 5)

**Idea -- distance as literal closeness.** The simplest notion of "close" is the one from geometry class: for two points a and b, take the difference on every axis, square it (so direction doesn't cancel out), add the squares up, and take the square root. In symbols, for features indexed by i:

$$d(a, b) = \sqrt{\sum_i (a_i - b_i)^2}$$

That's the whole formula -- the same straight-line ruler you'd use on a 2D map, just extended to five axes instead of two. Nothing about it needs training.

In [3]:
a = np.array(songs["Sunrise Drive"])
b = np.array(songs["Neon Nights"])

squared_diffs = [(ai - bi) ** 2 for ai, bi in zip(a, b)]
manual_distance = sum(squared_diffs) ** 0.5

print("squared differences per feature:", [round(v, 4) for v in squared_diffs])
print(f"by hand: sqrt({sum(squared_diffs):.4f}) = {manual_distance:.4f}")

squared differences per feature: [np.float64(0.0676), np.float64(0.0484), np.float64(0.0529), np.float64(0.0025), np.float64(0.0036)]
by hand: sqrt(0.1750) = 0.4183


**Idea -- same formula, vectorized.** Writing out a Python loop over five features is fine once. It stops being fine at catalogue scale. `numpy` computes the identical formula -- difference, square, sum, square root -- across every row of a whole matrix in one call, so distance-to-a-query can be measured against all twelve songs at once instead of one pair at a time.

In [4]:
def euclidean_distance_all(query, X):
    return np.linalg.norm(X - query, axis=1)

vector_distance = np.linalg.norm(a - b)
print(f"vectorized: {vector_distance:.4f}")
assert np.isclose(manual_distance, vector_distance), "by-hand and vectorized must match exactly"

dists = euclidean_distance_all(a, X_songs)
order = np.argsort(dists)
print("\ndistance from 'Sunrise Drive' to every song, nearest first:")
for i in order:
    print(f"  {dists[i]:.3f}  {song_names[i]}")

vectorized: 0.4183

distance from 'Sunrise Drive' to every song, nearest first:
  0.000  Sunrise Drive
  0.166  Road Trip Sing-Along
  0.308  Highway Chase
  0.418  Neon Nights
  0.472  Club Bassline
  0.483  Festival Anthem
  0.962  Coffee Shop Acoustic
  1.065  Slow Sunday
  1.122  Quiet Rain
  1.216  Midnight Study
  1.266  Piano Reflection
  3.977  Festival Anthem (Loud Remaster)


**Payoff.** The by-hand loop and the vectorized call agree to four decimal places (both 0.4183) -- the assert above would have failed loudly if they didn't, so this isn't a claim, it's a checked fact: `np.linalg.norm` is not a different idea, it's the identical formula, just faster. Ranking the whole catalogue by distance from "Sunrise Drive" puts the obviously similar upbeat tracks first (Road Trip Sing-Along at 0.166, Highway Chase at 0.308) and the quiet, acoustic tracks after them -- but the single farthest song of all twelve, at 3.977, is "Festival Anthem (Loud Remaster)": the loud twin of a song ("Festival Anthem," at 0.483) that itself reads as fairly close. Same underlying song, opposite verdicts, purely because one copy is loud. That's not a bug yet -- it's the first hint that this formula's "close" and a listener's "close" are not automatically the same thing. Distance, so far, is nothing more than arithmetic on the vectors from slide 5 -- there is no model here, no fitting, no parameters to learn.

## Part 2 -- Build kNN and Query It

**Problem.** Part 1 gives us a distance to every song from any one query, but a distance list isn't yet an answer to "find me five songs like this one" -- slide 7's actual question. We want a function that takes a query and a k, and hands back a ranked shortlist, the way a real retrieval system would.

**Ingredients.** A query vector that isn't any single song -- a listener's stated taste (high tempo, high energy, fairly danceable, not acoustic, upbeat) -- so the answer has to come from the catalogue, not from matching a row exactly.

In [5]:
query = np.array([0.80, 0.85, 0.70, 0.05, 0.60])
print("listener taste vector:", dict(zip(feature_names, query)))

listener taste vector: {'tempo': np.float64(0.8), 'energy': np.float64(0.85), 'danceability': np.float64(0.7), 'acousticness': np.float64(0.05), 'valence': np.float64(0.6)}


**Idea -- rank by distance, return the top k.** Slide 7 put the whole algorithm in one sentence: rank everything by distance, return the top k. There's no fitting step and nothing to remember between queries -- just measure, sort, slice.

In [6]:
def euclidean_neighbors(query, X, names, k=5):
    dists = np.linalg.norm(X - query, axis=1)
    order = np.argsort(dists)[:k]
    return [(names[i], float(dists[i])) for i in order]

print("Euclidean top 5 for the taste query:")
for name, dist in euclidean_neighbors(query, X_songs, song_names, k=5):
    print(f"  {dist:.3f}  {name}")

Euclidean top 5 for the taste query:
  0.130  Highway Chase
  0.198  Neon Nights
  0.238  Club Bassline
  0.245  Road Trip Sing-Along
  0.269  Sunrise Drive


**Payoff.** The top five (Highway Chase 0.130, Neon Nights 0.198, Club Bassline 0.238, Road Trip Sing-Along 0.245, Sunrise Drive 0.269) are exactly the upbeat, high-energy tracks a listener with this taste profile would expect, and the acoustic songs land at the bottom of the full ranking, over four times farther away. `euclidean_neighbors` is the entire retrieval engine from slide 7: no training data was withheld for it, no epochs were run, no accuracy was ever computed against a right answer -- because there isn't one. The generalizable point: retrieval doesn't need a training phase at all, only a query and a distance function, which is exactly why it still works the moment labels disappear.

**Idea -- see it, don't just read it.** Twelve songs of five numbers each don't plot on a page. Project down to two dimensions with PCA -- for looking only, the neighbour search above already ran in the true 5D space -- and turn k into something you can drag instead of retype.

In [7]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0).fit(X_songs)
X2 = pca.transform(X_songs)
query2 = pca.transform(query.reshape(1, -1))[0]

def plot_neighbours(k):
    nearest = {name for name, _ in euclidean_neighbors(query, X_songs, song_names, k=k)}
    colors = [RED if name in nearest else DIM for name in song_names]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=X2[:, 0], y=X2[:, 1], mode="markers+text", text=song_names,
                              textposition="top center", marker=dict(color=colors, size=10),
                              textfont=dict(size=9)))
    fig.add_trace(go.Scatter(x=[query2[0]], y=[query2[1]], mode="markers+text",
                              text=["query"], textposition="bottom center",
                              marker=dict(color=GOLD, size=16, symbol="star")))
    fig.update_layout(title=f"k = {k} nearest neighbours (PCA projection, for display only)",
                       width=700, height=500, showlegend=False)
    fig.show()

interact(plot_neighbours, k=IntSlider(min=1, max=11, step=1, value=5));

interactive(children=(IntSlider(value=5, description='k', max=11, min=1), Output()), _dom_classes=('widget-int…

**Payoff.** Drag k up: the highlighted set grows outward from the query star exactly like slide 7's circle widening. Watch where "Festival Anthem (Loud Remaster)" sits in this projection -- it stays grey (not selected) all the way until k=12, the entire catalogue, turning red only after every other song, including the quiet acoustic ones, has already been picked up. The takeaway: k isn't a property of the data, it's a knob you turn, and how many results count as "similar enough" is a decision the query-writer makes, not a fact retrieval discovers for you.

## Part 3 -- Euclidean vs Cosine: the Loud Remaster

**Problem.** Part 1 already showed a crack: the loud remaster of "Festival Anthem" sat oddly far from its own original for a supposedly identical song. Slide 8 named this directly -- Euclidean cares about magnitude, and turning up the volume moves every number, so a louder recording of the exact same song can measure as far away. We want to check that failure with real numbers, and see whether a different distance sees through it.

**Ingredients.** No new data -- reuse `X_songs`, including the planted "Festival Anthem (Loud Remaster)" row from Part 1. This time we query with "Festival Anthem" itself, so the honestly correct answer is obvious: its own remaster should come back first.

**Idea -- distance as direction, not magnitude.** Cosine similarity ignores length entirely and measures only the angle between two vectors: divide each vector by its own length first, so every song becomes a "unit" vector pointing somewhere, then take the dot product.

$$\cos(a, b) = \frac{a}{\lVert a \rVert} \cdot \frac{b}{\lVert b \rVert}$$

Scaling a vector -- turning the volume up on every feature by the same factor -- moves its length but not its direction, so cosine similarity between a song and a louder recording of that same song should land at (or very near) 1.0: pointing the same way.

In [8]:
def cosine_neighbors(query, X, names, k=5):
    q_unit = query / np.linalg.norm(query)
    X_unit = X / np.linalg.norm(X, axis=1, keepdims=True)
    similarities = X_unit @ q_unit
    order = np.argsort(-similarities)[:k]
    return [(names[i], float(similarities[i])) for i in order]

anthem = np.array(songs["Festival Anthem"])

print("Euclidean ranking, query = 'Festival Anthem' (full catalogue):")
for name, dist in euclidean_neighbors(anthem, X_songs, song_names, k=len(song_names)):
    print(f"  {dist:.3f}  {name}")

print("\nCosine ranking, same query:")
for name, sim in cosine_neighbors(anthem, X_songs, song_names, k=len(song_names)):
    print(f"  {sim:.4f}  {name}")

Euclidean ranking, query = 'Festival Anthem' (full catalogue):
  0.000  Festival Anthem
  0.096  Neon Nights
  0.135  Club Bassline
  0.358  Road Trip Sing-Along
  0.367  Highway Chase
  0.483  Sunrise Drive
  1.361  Coffee Shop Acoustic
  1.478  Slow Sunday
  1.532  Quiet Rain
  1.630  Midnight Study
  1.684  Piano Reflection
  3.508  Festival Anthem (Loud Remaster)

Cosine ranking, same query:
  1.0000  Festival Anthem (Loud Remaster)
  1.0000  Festival Anthem
  0.9994  Neon Nights
  0.9974  Club Bassline
  0.9955  Sunrise Drive
  0.9931  Road Trip Sing-Along
  0.9893  Highway Chase
  0.6319  Coffee Shop Acoustic
  0.5387  Slow Sunday
  0.4913  Quiet Rain
  0.4007  Midnight Study
  0.3499  Piano Reflection


**Payoff.** Querying with "Festival Anthem" itself should trivially return its own louder remaster first -- and Euclidean gets this backwards in the most dramatic way possible: the remaster ranks dead last of all twelve songs, at distance 3.508, farther away than every quiet acoustic track in the catalogue. Cosine gets it right: the remaster ties for first place at similarity 1.0000, exactly level with the unscaled original, because tripling every feature moves a vector's length, not its direction. Same query, same catalogue, opposite answers -- and nobody handed either metric to us as "the" right one. That's slide 8's point made concrete: choosing Euclidean or cosine is choosing what "similar" is allowed to mean, and that choice belongs to whoever writes the query, not to the data.

## Challenge -- break it deliberately: duration in milliseconds

**Problem.** Every feature so far has lived on the same friendly 0-1 scale, by design. Real catalogues never arrive that clean -- a streaming service's raw song table would also carry things like duration in milliseconds, values in the hundreds of thousands. Will `euclidean_neighbors`, unmodified, survive one honestly-scaled real-world feature added to the mix?

**Ingredients.** A duration in milliseconds for each song (roughly 3 to 4 minutes -- 198,000 to 260,000 ms -- except "Highway Chase," deliberately built as a 5:50 outlier at 350,000 ms), appended as a sixth feature.

In [9]:
durations_ms = {
    "Sunrise Drive": 210000, "Neon Nights": 198000, "Quiet Rain": 245000,
    "Midnight Study": 260000, "Coffee Shop Acoustic": 230000, "Festival Anthem": 205000,
    "Highway Chase": 350000, "Slow Sunday": 222000, "Club Bassline": 215000,
    "Piano Reflection": 255000, "Road Trip Sing-Along": 208000,
    "Festival Anthem (Loud Remaster)": 205000,
}
duration_col = np.array([durations_ms[n] for n in song_names]).reshape(-1, 1)
X_aug = np.hstack([X_songs, duration_col])
print("augmented feature matrix shape:", X_aug.shape)

augmented feature matrix shape: (12, 6)


**Idea -- run the identical function, no code changes.** `euclidean_neighbors` doesn't know or care what a feature means -- it just measures differences. Query it with the same taste vector as Part 2, plus a plausible duration (210,000 ms, a 3.5-minute guess), and see whether the ranking still makes musical sense.

In [10]:
query_aug = np.append(query, 210000.0)

print("Euclidean top 5, taste + raw duration (ms):")
for name, dist in euclidean_neighbors(query_aug, X_aug, song_names, k=5):
    print(f"  {dist:.1f}  {name}")

print("\nfor comparison, Part 2's top 5 without duration was:")
print("  Highway Chase, Neon Nights, Club Bassline, Road Trip Sing-Along, Sunrise Drive")

Euclidean top 5, taste + raw duration (ms):
  0.3  Sunrise Drive
  2000.0  Road Trip Sing-Along
  5000.0  Club Bassline
  5000.0  Festival Anthem
  5000.0  Festival Anthem (Loud Remaster)

for comparison, Part 2's top 5 without duration was:
  Highway Chase, Neon Nights, Club Bassline, Road Trip Sing-Along, Sunrise Drive


**Payoff -- your answer (2-3 sentences).** Look at the numbers above: Highway Chase, Part 2's single best match by taste, doesn't even appear in this top 5 anymore, while Sunrise Drive jumps from fifth place to first. In your own words, why did adding one feature on a scale of hundreds of thousands overturn a ranking that four carefully-matched 0-1 features had already agreed on?

*(write your answer here)*

**Idea -- put every feature on the same scale.** The fix isn't a new algorithm -- `euclidean_neighbors` is unchanged from Part 1, Part 2, and here. The fix is the representation: rescale every column, including duration, onto a comparable range before measuring distance, so no single feature's raw units can dominate just because it happens to be counted in milliseconds instead of fractions.

In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler().fit(X_aug)
X_scaled = scaler.transform(X_aug)
query_scaled = scaler.transform(query_aug.reshape(1, -1))[0]

print("Euclidean top 5, all six features scaled to [0, 1]:")
for name, dist in euclidean_neighbors(query_scaled, X_scaled, song_names, k=5):
    print(f"  {dist:.3f}  {name}")

Euclidean top 5, all six features scaled to [0, 1]:
  0.109  Club Bassline
  0.115  Neon Nights
  0.118  Sunrise Drive
  0.127  Road Trip Sing-Along
  0.130  Festival Anthem


**Payoff.** Scaling recovers Club Bassline, Neon Nights, Sunrise Drive, and Road Trip Sing-Along -- four of Part 2's original top five, now with duration folded in honestly instead of dominating. Highway Chase still doesn't return: its 350,000 ms runtime really is a genuine outlier, over 100 seconds longer than anything else in the catalogue, and even a fairly-weighted duration difference now correctly counts against it. Nothing here was a bug to fix once and forget -- which features to include, and what scale to measure them on, is exactly the choice slide 8 said the label's disappearance handed to you. The representation is a decision you own, every time, not a default that comes free with the data.

## Reflection

**Problem.** You now own the full retrieval toolkit: turning rows into vectors, measuring distance two different ways, building the rank-and-slice retrieval function, and watching the representation itself decide the answer. The point of this last cell isn't a right answer -- it's stating, in your own words, the reasoning each part was built on.

In [12]:
# 1. Part 1 showed the by-hand loop and np.linalg.norm agree exactly (both
#    0.4183). What does that tell you about what a "distance function"
#    actually is -- and why does it need no training data to compute?
answer_1 = """
"""

# 2. Part 2's euclidean_neighbors function had no fit step, no epochs,
#    nothing learned between queries. Contrast that with a supervised model
#    you've trained before: what is retrieval doing instead of learning?
answer_2 = """
"""

# 3. Part 3 showed Euclidean and cosine give the same query, "Festival
#    Anthem," opposite answers about its own loud remaster. Explain in your
#    own words what each metric actually measures, and why they disagree
#    specifically on a scaled duplicate.
answer_3 = """
"""

# 4. The Challenge showed that scaling recovered most, but not all, of Part
#    2's original top 5 -- Highway Chase never came back. Why should a
#    genuinely large duration difference still matter even after scaling,
#    when it dominated everything before scaling?
answer_4 = """
"""

# 5. If somebody handed you an unlabelled dataset tomorrow, name the two
#    decisions this lab forced you to make before any retrieval could run
#    at all -- and explain why nobody could have made either decision for
#    you.
answer_5 = """
"""

---
*Retrieval: Finding Similar Items Without Labels -- Bluevision-ai Academy*